<a href="https://colab.research.google.com/github/0sinach1/house-price-model/blob/main/EDA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# House Price Prediction in Nigeria

This notebook covers exploratory data analysis, preprocessing, modeling, and evaluation for predicting house prices in Nigeria.


## 1. Import Required Libraries

We will use pandas, numpy, matplotlib, seaborn, and scikit-learn for data analysis and modeling.

In [4]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [7]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## 2. Load and Explore the Dataset

Load the Nigeria house prices dataset and display basic information.

In [8]:
# Load the dataset
file_path = "/content/drive/MyDrive/Datasets/nigeria_houses_data.csv"
df = pd.read_csv(file_path)
df.head()

,bedrooms,bathrooms,toilets,parking_space,title,town,state,price
0,6.0,5.0,5.0,4.0,Detached Duplex,Mabushi,Abuja,450000000.0
1,4.0,5.0,5.0,4.0,Terraced Duplexes,Katampe,Abuja,800000000.0
2,4.0,5.0,5.0,4.0,Detached Duplex,Lekki,Lagos,120000000.0
3,4.0,4.0,5.0,6.0,Detached Duplex,Ajah,Lagos,40000000.0
4,4.0,4.0,5.0,2.0,Semi Detached Duplex,Lekki,Lagos,75000000.0


In [9]:

print("Creating new features...")

# 1. Price per square meter (if you have sqm column)
# Check if 'sqm' or 'size' or 'area' column exists
if 'sqm' in df.columns:
    df['price_per_sqm'] = df['price'] / df['sqm']
    print("✅ Created: price_per_sqm")
elif 'size' in df.columns:
    df['price_per_sqm'] = df['price'] / df['size']
    print("✅ Created: price_per_sqm")
else:
    print("⚠️  No sqm/size column found - skipping price_per_sqm")

# 2. Bedroom to bathroom ratio
df['bedroom_bathroom_ratio'] = df['bedrooms'] / (df['bathrooms'] + 0.1)  # +0.1 to avoid division by zero
print("✅ Created: bedroom_bathroom_ratio")

# 3. Total rooms (bedrooms + bathrooms + toilets)
df['total_rooms'] = df['bedrooms'] + df['bathrooms'] + df['toilets']
print("✅ Created: total_rooms")

# 4. Luxury score (sum of premium features)
df['luxury_score'] = df['parking_space'].clip(upper=5)  # Cap at 5
print("✅ Created: luxury_score")

# 5. Is Lagos (binary feature - Lagos properties are typically more expensive)
df['is_lagos'] = (df['state'] == 'Lagos').astype(int)
print("✅ Created: is_lagos")

# 6. Is Abuja (second most expensive)
df['is_abuja'] = (df['state'] == 'Abuja').astype(int)
print("✅ Created: is_abuja")

# 7. Is premium location (Lagos or Abuja)
df['is_premium_location'] = ((df['state'] == 'Lagos') | (df['state'] == 'Abuja')).astype(int)
print("✅ Created: is_premium_location")

# Display new features
print("\n" + "="*60)
print("NEW FEATURES CREATED")
print("="*60)
print(df[['bedrooms', 'bathrooms', 'bedroom_bathroom_ratio',
          'total_rooms', 'luxury_score', 'is_lagos', 'is_abuja']].head())
print("\n✅ Feature engineering complete!")

Creating new features...
⚠️  No sqm/size column found - skipping price_per_sqm
✅ Created: bedroom_bathroom_ratio
✅ Created: total_rooms
✅ Created: luxury_score
✅ Created: is_lagos
✅ Created: is_abuja
✅ Created: is_premium_location

NEW FEATURES CREATED
   bedrooms  bathrooms  bedroom_bathroom_ratio  total_rooms  luxury_score  \
0       6.0        5.0                1.176471         16.0           4.0   
1       4.0        5.0                0.784314         14.0           4.0   
2       4.0        5.0                0.784314         14.0           4.0   
3       4.0        4.0                0.975610         13.0           5.0   
4       4.0        4.0                0.975610         13.0           2.0   

   is_lagos  is_abuja  
0         0         1  
1         0         1  
2         1         0  
3         1         0  
4         1         0  

✅ Feature engineering complete!


### 2.1 Data Structure and Missing Values

Check the dataframe's shape, info, and count missing values.

In [ ]:
# DataFrame info and shape
print("Shape:", df.shape)
df.info()

In [ ]:
# Count missing values for each column
df.isnull().sum()

### 2.2 Visualize Target Variable Distribution

Plot the distribution of the 'price' column and its log-transformed version.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['price'], bins=30, kde=True)
plt.title('Original Price Distribution')
plt.xlabel('Price')

plt.subplot(1, 2, 2)
sns.histplot(np.log1p(df['price']), bins=30, kde=True)
plt.title('Log-Transformed Price Distribution')
plt.xlabel('log(Price)')
plt.tight_layout()
plt.show()

### 2.3 Analyze Categorical Features

Display the most common states in the dataset.

In [ ]:
# Top 10 states by count
state_counts = df['state'].value_counts().head(10)
print(state_counts)

## 3. Data Preprocessing

Prepare data for modeling: define features and target, then split into train/test sets.

In [ ]:
target_col = 'price'

In [ ]:
exclude = ['id'] if 'id' in df.columns else []
feature_cols = [c for c in df.columns if c not in exclude + [target_col]]

X = df[feature_cols].copy()
y = df[target_col].copy()

### 3.1 Train-Test Split

Split data into training (80%) and testing (20%) sets to prevent data leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

### 3.2 Identify Numeric and Categorical Columns

Separate features by data type for appropriate preprocessing.

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols),
], remainder='drop')


## 5. Train the Model

Create and train a Ridge Regression model with log-transformed target.

In [ ]:
y_train_log = np.log1p(y_train)

model = Pipeline([
    ('preproc', preprocessor),
    ('reg', LinearRegression())
])

## 6. Evaluate Model Performance

Generate predictions and calculate performance metrics on the test set.

In [ ]:
model.fit(X_train, y_train_log)

In [ ]:
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)

In [ ]:
from sklearn.linear_model import Ridge

model = Pipeline([
    ('preproc', preprocessor),
    ('reg', Ridge(alpha=1.0))  # Regularization reduces extreme predictions
])

model.fit(X_train, y_train_log)
y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"RMSE (original units): {rmse:.2f}")
print(f"R^2: {r2:.3f}")

### 6.1 Cross-Validation Score

Validate model performance using 5-fold cross-validation.

In [ ]:
cv_scores = cross_val_score(model, X_train, y_train_log, scoring='neg_root_mean_squared_error', cv=5)
cv_rmse = -cv_scores.mean()
cv_std = cv_scores.std()

print("=" * 50)
print("CROSS-VALIDATION RESULTS (5-Fold)")
print("=" * 50)
print(f"Mean CV RMSE (log-target): {cv_rmse:.4f}")
print(f"Std Dev: {cv_std:.4f}")
print("=" * 50)

### 6.2 Residual Analysis

Analyze model residuals to check for patterns and assumptions.

In [ ]:
residuals = y_test - y_pred
plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(0, color='r', linestyle='--')
plt.xlabel('Predicted Price')
plt.ylabel('Residuals')
plt.title('Residual Plot')

plt.subplot(1, 2, 2)
sns.histplot(residuals, bins=30, kde=True)
plt.xlabel('Residuals')
plt.title('Residual Distribution')
plt.tight_layout()
plt.show()

## 7. Conclusion and Next Steps

We successfully built a house price prediction model using Ridge Regression with proper preprocessing pipelines. The model was trained on log-transformed prices to handle the skewed distribution.

### Key Achievements:
- ✅ Proper data leakage prevention (train/test split before preprocessing)
- ✅ Robust handling of missing values and categorical encoding
- ✅ Log-transformation to normalize target variable
- ✅ Cross-validation for stable performance estimation
- ✅ Residual analysis for model diagnostics

### Future Improvements:
- Try advanced models (Random Forest, XGBoost, Gradient Boosting)
- Hyperparameter tuning (Grid Search / Random Search)
- Feature importance analysis
- Outlier detection and handling
- Target encoding for high-cardinality categorical features